# Task 2: Prediction Artifact Audit for Paired Bootstrap

Purpose: before running bootstrap confidence intervals, this notebook checks whether the existing milestone outputs contain raw prediction artifacts.

Raw predictions are required because bootstrap needs paired rows: the same `y_true` and comparable model `score` for the reference and perturbed scenarios.

Run this notebook first. If it reports `READY_FOR_BOOTSTRAP = False`, do not start bootstrap yet. Add the export helper from the final cells to the relevant experiment notebooks and rerun only the needed experiments.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import re
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 160)

## 1. Configure Drive paths

This matches your Drive screenshot: `My Drive > Credit__ > Datasets`.

In [ ]:
PROJECT_ROOT = Path('/content/drive/MyDrive/Credit__')

MILESTONE_DIRS = {
    '03_leakage_stress_testing': PROJECT_ROOT / '03_leakage_stress_testing',
    '06_temporal_robustness': PROJECT_ROOT / '06_temporal_robustness',
    '07_prevalence_stress_test': PROJECT_ROOT / '07_prevalence_stress_test',
    '08_calibration_recalibration': PROJECT_ROOT / '08_calibration_recalibration',
    '09_explanation_stability': PROJECT_ROOT / '09_explanation_stability',
}

FORMALIZATION_DIR = PROJECT_ROOT / '10_framework_formalization'
RESULTS_DIR = FORMALIZATION_DIR / 'results'
PREDICTION_EXPORT_DIR = FORMALIZATION_DIR / 'prediction_exports'
AUDIT_DIR = FORMALIZATION_DIR / 'task2_prediction_artifact_audit'

for path in [FORMALIZATION_DIR, RESULTS_DIR, PREDICTION_EXPORT_DIR, AUDIT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROJECT_ROOT exists:', PROJECT_ROOT.exists())
for name, path in MILESTONE_DIRS.items():
    print(f'{name}: {path.exists()} -> {path}')

## 2. Required prediction schema

Minimum required columns for bootstrap:

`dataset`, `model`, `scenario`, `seed`, `split`, `row_id`, `y_true`, `score`, `threshold`

The audit accepts common aliases such as `label`, `Class`, `isFraud`, `probability`, `y_score`, and `prediction_score`, but the clean export should use the exact standard names above.

In [ ]:
REQUIRED_STANDARD_COLUMNS = [
    'dataset', 'model', 'scenario', 'seed', 'split', 'row_id', 'y_true', 'score', 'threshold'
]

COLUMN_ALIASES = {
    'dataset': {'dataset', 'dataset_name', 'data', 'corpus'},
    'model': {'model', 'model_name', 'classifier', 'estimator'},
    'scenario': {'scenario', 'split_policy', 'condition', 'protocol', 'calibrator'},
    'seed': {'seed', 'random_seed'},
    'split': {'split', 'partition', 'fold'},
    'row_id': {'row_id', 'index', 'sample_id', 'transaction_id', 'idx'},
    'y_true': {'y_true', 'true_label', 'label', 'target', 'class', 'Class', 'isFraud', 'is_fraud'},
    'score': {'score', 'scores', 'y_score', 'probability', 'prob_fraud', 'prediction_score', 'test_scores', 'pred_proba'},
    'threshold': {'threshold', 'tau', 'decision_threshold'},
}

LOWER_ALIAS_LOOKUP = {
    alias.lower(): standard
    for standard, aliases in COLUMN_ALIASES.items()
    for alias in aliases
}

def canonical_column_map(columns):
    mapping = {}
    used = set()
    for original in columns:
        standard = LOWER_ALIAS_LOOKUP.get(str(original).lower())
        if standard and standard not in used:
            mapping[original] = standard
            used.add(standard)
    return mapping

def standardize_preview_columns(df):
    mapping = canonical_column_map(df.columns)
    return df.rename(columns=mapping), mapping

## 3. Scan CSV artifacts

This scan intentionally ignores `Datasets/` so it does not accidentally read huge raw datasets. It reads only a few rows from CSV files in milestone output folders.

In [ ]:
def iter_csv_artifacts():
    roots = list(MILESTONE_DIRS.items()) + [
        ('10_framework_formalization', FORMALIZATION_DIR),
    ]
    for milestone, root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob('*.csv')):
            parts_lower = {p.lower() for p in path.parts}
            if 'datasets' in parts_lower:
                continue
            yield milestone, path

def safe_read_preview(path, nrows=25):
    try:
        return pd.read_csv(path, nrows=nrows), None
    except Exception as exc:
        return None, f'{type(exc).__name__}: {exc}'

def infer_file_role(path, columns):
    name = path.name.lower()
    cols = {str(c).lower() for c in columns}
    if any(token in name for token in ['pred', 'prob', 'score']) and not any(token in name for token in ['importance', 'summary', 'index']):
        return 'possible_prediction_file'
    if {'average_precision', 'mcc'} & cols or {'roc_auc', 'pr_auc'} & cols:
        return 'metric_checkpoint_or_result'
    if {'feature', 'importance', 'rank'} & cols:
        return 'importance_or_explanation_artifact'
    if {'ece', 'brier_score'} & cols or 'calibration' in name:
        return 'calibration_metric_or_bins'
    return 'other_csv'

def audit_one_csv(milestone, path):
    df, error = safe_read_preview(path)
    row = {
        'milestone': milestone,
        'path': str(path),
        'file_name': path.name,
        'read_ok': error is None,
        'read_error': error,
        'columns': '',
        'canonical_columns_found': '',
        'missing_required_columns': ','.join(REQUIRED_STANDARD_COLUMNS),
        'has_label_column': False,
        'has_score_column': False,
        'has_pairing_columns': False,
        'candidate_status': 'unreadable',
        'file_role': 'unreadable',
    }
    if error is not None:
        return row

    standardized, mapping = standardize_preview_columns(df)
    found = sorted(set(standardized.columns) & set(REQUIRED_STANDARD_COLUMNS))
    missing = [c for c in REQUIRED_STANDARD_COLUMNS if c not in found]
    row.update({
        'columns': '|'.join(map(str, df.columns)),
        'canonical_columns_found': '|'.join(found),
        'missing_required_columns': '|'.join(missing),
        'has_label_column': 'y_true' in found,
        'has_score_column': 'score' in found,
        'has_pairing_columns': all(c in found for c in ['dataset', 'model', 'scenario', 'seed']),
        'file_role': infer_file_role(path, df.columns),
    })

    if not missing:
        row['candidate_status'] = 'PASS_RAW_PREDICTION_SCHEMA'
    elif row['has_label_column'] and row['has_score_column']:
        row['candidate_status'] = 'PARTIAL_LABEL_AND_SCORE_PRESENT'
    elif row['file_role'] == 'metric_checkpoint_or_result':
        row['candidate_status'] = 'METRICS_ONLY_NOT_BOOTSTRAP_READY'
    elif row['file_role'] == 'possible_prediction_file':
        row['candidate_status'] = 'PREDICTION_NAMED_BUT_SCHEMA_INCOMPLETE'
    else:
        row['candidate_status'] = 'NOT_A_RAW_PREDICTION_FILE'
    return row

audit_rows = [audit_one_csv(milestone, path) for milestone, path in iter_csv_artifacts()]
audit = pd.DataFrame(audit_rows)

audit_path = RESULTS_DIR / 'prediction_artifact_audit.csv'
audit.to_csv(audit_path, index=False)

print('Scanned CSV files:', len(audit))
print('Audit saved to:', audit_path)
display(audit['candidate_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='n'))
display(audit[['milestone', 'file_name', 'candidate_status', 'file_role', 'canonical_columns_found', 'missing_required_columns']].head(30))

## 4. Bootstrap readiness gate

Pass condition: at least one valid raw prediction CSV exists, and every expected milestone has either valid prediction exports or an explicit reason why it must be rerun/exported.

In [ ]:
valid_predictions = audit[audit['candidate_status'].eq('PASS_RAW_PREDICTION_SCHEMA')].copy()
partial_predictions = audit[audit['candidate_status'].eq('PARTIAL_LABEL_AND_SCORE_PRESENT')].copy()
metrics_only = audit[audit['candidate_status'].eq('METRICS_ONLY_NOT_BOOTSTRAP_READY')].copy()

READY_FOR_BOOTSTRAP = len(valid_predictions) > 0

print('READY_FOR_BOOTSTRAP =', READY_FOR_BOOTSTRAP)
print('Valid raw prediction files:', len(valid_predictions))
print('Partial label+score files:', len(partial_predictions))
print('Metrics-only files:', len(metrics_only))

if len(valid_predictions):
    display(valid_predictions[['milestone', 'file_name', 'path', 'canonical_columns_found']].head(50))

if len(partial_predictions):
    print('\nPartial candidates that need standard columns added:')
    display(partial_predictions[['milestone', 'file_name', 'path', 'canonical_columns_found', 'missing_required_columns']].head(50))

if not READY_FOR_BOOTSTRAP:
    print('\nNo complete raw prediction artifact found.')
    print('Next action: add the export helper below into each experiment notebook, rerun the experiment cells, then rerun this audit notebook.')

## 5. Milestone-level summary

This table tells you exactly which milestone is ready and which one is only metrics-level evidence.

In [ ]:
summary = (
    audit
    .groupby(['milestone', 'candidate_status'])
    .size()
    .reset_index(name='n_files')
    .sort_values(['milestone', 'candidate_status'])
)
summary_path = RESULTS_DIR / 'prediction_artifact_audit_summary.csv'
summary.to_csv(summary_path, index=False)
display(summary)
print('Summary saved to:', summary_path)

## 6. Standard prediction export helper

Add this helper near the utility/helper section of every experiment notebook, after `write_csv_atomic` is defined and after `EXPERIMENT_ROOT` is created.

Then call `save_prediction_export(...)` immediately after `test_scores = get_scores(...)` and before/after metrics are computed.

`python
PREDICTION_EXPORT_DIR = EXPERIMENT_ROOT / 'prediction_exports'
PREDICTION_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def _safe_name(value):
    value = str(value)
    value = re.sub(r'[^A-Za-z0-9_.-]+', '_', value)
    return value.strip('_')

def save_prediction_export(
    dataset,
    model_name,
    scenario,
    seed,
    split,
    y_true,
    scores,
    threshold,
    row_id=None,
    extra_columns=None,
):
    y_arr = np.asarray(y_true).reshape(-1)
    s_arr = np.asarray(scores).reshape(-1)
    if len(y_arr) != len(s_arr):
        raise ValueError(f'y_true length {len(y_arr)} != score length {len(s_arr)}')

    if row_id is None:
        row_arr = np.arange(len(y_arr))
    else:
        row_arr = np.asarray(row_id).reshape(-1)
        if len(row_arr) != len(y_arr):
            raise ValueError(f'row_id length {len(row_arr)} != y_true length {len(y_arr)}')

    out = pd.DataFrame({
        'dataset': str(dataset),
        'model': str(model_name),
        'scenario': str(scenario),
        'seed': int(seed),
        'split': str(split),
        'row_id': row_arr,
        'y_true': y_arr.astype(int),
        'score': s_arr.astype(float),
        'threshold': float(threshold),
    })

    if extra_columns:
        for key, value in extra_columns.items():
            if np.isscalar(value) or isinstance(value, str):
                out[str(key)] = value
            else:
                value_arr = np.asarray(value)
                if len(value_arr) != len(out):
                    raise ValueError(f'extra column {key} length {len(value_arr)} != output length {len(out)}')
                out[str(key)] = value_arr

    file_name = (
        f"seed{int(seed)}__{_safe_name(dataset)}__{_safe_name(scenario)}__"
        f"{_safe_name(model_name)}__{_safe_name(split)}__predictions.csv"
    )
    path = PREDICTION_EXPORT_DIR / file_name
    write_csv_atomic(path, out)
    return path
`


## 7. Exact call pattern for each old notebook

Use the matching call after `test_scores = get_scores(...)`.

`python
# Leakage stress / multiseed leakage
prediction_path = save_prediction_export(
    dataset=dataset_name,
    model_name=model_name,
    scenario=scenario,
    seed=seed,
    split='test',
    y_true=y_test,
    scores=test_scores,
    threshold=tau,
)
print('Saved predictions:', prediction_path)
`


`python
# Temporal robustness
prediction_path = save_prediction_export(
    dataset=dataset_name,
    model_name=model_name,
    scenario=split_policy,
    seed=seed,
    split='test',
    y_true=y_test,
    scores=test_scores,
    threshold=tau,
)
print('Saved predictions:', prediction_path)
`


`python
# Prevalence stress
# Save the natural test stream prediction once, before prevalence resampling.
prediction_path = save_prediction_export(
    dataset=dataset_name,
    model_name=model_name,
    scenario='NATURAL_TEST_STREAM',
    seed=seed,
    split='test',
    y_true=y_test,
    scores=trained['test_scores'],
    threshold=trained['threshold'],
)
print('Saved predictions:', prediction_path)
`


`python
# Calibration/recalibration
# Put this inside the calibrator loop, after test_scores is produced.
prediction_path = save_prediction_export(
    dataset=dataset_name,
    model_name=model_name,
    scenario=f'CALIBRATION_{calibrator_name}',
    seed=seed,
    split='test',
    y_true=y_test.reset_index(drop=True),
    scores=test_scores,
    threshold=threshold,
)
print('Saved predictions:', prediction_path)
`


`python
# Explanation stability
# Use the split policy as the scenario name.
prediction_path = save_prediction_export(
    dataset=dataset_name,
    model_name=model_name,
    scenario=split_policy,
    seed=seed,
    split='test',
    y_true=y_test,
    scores=test_scores,
    threshold=tau,
)
print('Saved predictions:', prediction_path)
`


## 8. After rerunning old notebooks

After the old notebooks save prediction CSV files:

1. Rerun this audit notebook from the top.
2. Confirm `READY_FOR_BOOTSTRAP = True`.
3. Confirm `prediction_artifact_audit.csv` contains `PASS_RAW_PREDICTION_SCHEMA` rows.
4. Only then start paired bootstrap.